In [1]:
%config Completer.use_jedi = False

In [3]:
# load module
import os
#os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorboard as tb 
import pytorch_lightning as pl


from pathlib import Path
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet , EncoderNormalizer , GroupNormalizer
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
from IPython.core.display import HTML

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'
#plt.rcParams['font.sans-serif'] = ['NanumGothic.ttf', 'sans-serif']

from my_funs import *

In [4]:
epochs = [8, 16, 32 ,48 ,  64, 80]


from itertools import product
result_df = pd.DataFrame(epochs)
result_df.columns = ['hidden_size']
result_df['model_ckpt'] = ""
result_df

,hidden_size,model_ckpt
0,8,
1,16,
2,32,
3,48,
4,64,
5,80,


## 다음과 같은 setting에서 진행 

ewma, factor , lr , optim은 최적의 값으로 진행  

| ewma | factor | lr | optim | batch_size | epoch | atten_head |
|---|---|---|---|---|---|---|
| alpha | 0.2 | 0.01 | ranger | 32 | 100 | 4 |


In [5]:
# 노트북 꺼지면 이걸로 불러오기 
#result_df = pd.read_csv('hiddne_size_setting.csv')

In [6]:
# ewma, lr, optim은 이전 TEST에서 구한 최적의 값 사용
pl.seed_everything(42)
batch_size = 32

for idx in result_df.index:
    if result_df.loc[idx , 'model_ckpt'] == "":
        hidden_size = int(result_df['hidden_size'].loc[idx])

        data = data_processing('../../train.csv' , 0.2 ,moving_average_alpha )
        training = get_training(data,24, 24*7)

        dataloader = training.to_dataloader(batch_size)
        validation = TimeSeriesDataSet.from_dataset(training, data, predict=True, stop_randomization=True)
        train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
        val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size*100, num_workers=0)

        early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
        lr_logger = LearningRateMonitor()  
        logger = TensorBoardLogger('Hidden_size_model_ckpt')
        trainer = pl.Trainer(
            max_epochs=100,
            gpus=1,
            gradient_clip_val=0.1,
            limit_train_batches=20,  
            callbacks=[lr_logger],
            logger=logger,
        )


        tft = TemporalFusionTransformer.from_dataset(
            training,
            learning_rate=0.01,
            hidden_size=hidden_size ,#hidden,  # most important hyperparameter apart from learning rate
            attention_head_size=4,
            dropout=0.1,  # between 0.1 and 0.3 are good values
            hidden_continuous_size=hidden_size,#hidden,  # set to <= hidden_size
            output_size=7,  # 7 quantiles by default
            loss=QuantileLoss(),
            # reduce learning rate if no improvement in validation loss after x epochs
            reduce_on_plateau_patience=4,
        )

        tft.hparams['optimizer'] = 'ranger'

        trainer.fit(
        tft,
        train_dataloaders=train_dataloader,
        val_dataloaders=val_dataloader
        )

        print(trainer.checkpoint_callback.best_model_path)
        
        result_df.loc[idx, 'model_ckpt'] = trainer.checkpoint_callback.best_model_path
        result_df.to_csv('hidden_size_setting.csv')

Global seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: Hidden_size_model_ckpt/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 493   
3  | prescalers                         | ModuleDict                      | 112   
4  | static_variable_selection          | VariableSelectionNetwork        | 406   
5  | encoder_variable_selection         | VariableSelectionNetwork        | 3.0 K 
6  | decoder_variable_selection         | VariableSe

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


Hidden_size_model_ckpt/lightning_logs/version_5/checkpoints/epoch=99-step=2000.ckpt
